# シミュレーション計算実行用ノートブック

## 1. 今回の実験の説明

In [ ]:
DESCRIPTION = '''
mlflowを使ってシミュレーションプログラムの実行とその結果を管理する実験の例題1。DEM5のシミュレーションを行う。
'''
ISSUE_NO = ''
EXEC_NAME = 'DEM5'
RUN_SCRIPT = "calcDEM5"

PREV_RUNID = ''
RESTART_FROM = ''

MLFLOW_EXP_TYPE = "particleDEM"

USE_MPI = False
MPI_NP = 4

## 2. シミュレーションパラメータ

In [ ]:
import importlib, json, math
sim = importlib.import_module(f'scripts.{RUN_SCRIPT}')

sim_params = sim.default_prams

## 必要なら適宜修正する
sim_params["total_time"] = 1.5

print(json.dumps(sim_params, indent=4, ensure_ascii=False))

## artifacts として回収する
dump_files = ["dump.DEM5", "dump.DEM5box"]
snapshots = "dump.DEM5box"
restart_files = []

artifacts_cleanup = True

## 3. mlflow変数

In [ ]:
import mlflow
import os
from time import strftime, gmtime

In [ ]:
ROOT = os.getenv("HOME")

## 1台構成の時
#MLFLOW_TRACKING_URI = f"sqlite:///{ROOT}/mlruns/mlflow.db"
#MLFLOW_STORAGE = f"file://{ROOT}/mlstorage"
### S3 bucket を指定する場合
MLFLOW_STORAGE = f"s3://{os.getenv('S3STORAGEBUCKET', 'my-mlflow-artifact-s3-bucket')}/mlstorage/"

## mlflow serverのIPを指定
MLFLOW_TRACKING_URI = "http://localhost:5000"


## github, backlogなどでチケット管理をしている場合はそのBASE URLを設定
ISSUE_BASE_URL = 'https://xxxxx/'

### dump fileの圧縮に使うコマンド（pixz があれば推奨）
ARCHIVE_COMMAND = "pixz"

###
### mlflow変数　自動設定
###
MYNAME = os.getenv("USER")
#GIT_INFO = gitutils.get_info()
RUN_NAME = EXEC_NAME + strftime("-%Y-%m-%d-%H-%M-%S", gmtime())

if PREV_RUNID != '':
    prev_run = mlflow.get_run(PREV_RUNID)
    prev_experiment_id = prev_run.info.experiment_id
    tracking_uri = mlflow.get_tracking_uri().rstrip("/")
    prev_url = f"{tracking_uri}/#/experiments/{prev_experiment_id}/runs/{PREV_RUNID}"
    
    restart = f"（{RESTART_FROM}）" if RESTART_FROM != '' else ''

    FORMER_EXP = f"\n[PREV_RUNID]({prev_url})より派生{restart}"
else:
    FORMER_EXP = ""

ISSUE_NAME = f'\n[{ISSUE_NO}]({ISSUE_BASE_URL}{ISSUE_NO})' if ISSUE_NO != '' else ''

## 4. シミュレーション実行

In [ ]:
###
### mlflow処理開始
###
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow_exp = mlflow.get_experiment_by_name(MLFLOW_EXP_TYPE)
if mlflow_exp is None:
    mlflow_exp_id = mlflow.create_experiment(name=MLFLOW_EXP_TYPE, artifact_location=MLFLOW_STORAGE)
else:
    mlflow_exp_id = mlflow_exp.experiment_id

In [ ]:
mlflow_run = mlflow.start_run(
    experiment_id=mlflow_exp_id,
    run_name=RUN_NAME,
    description=f'{DESCRIPTION}{FORMER_EXP}{ISSUE_NAME}')

print(f"Run ID: {mlflow_run.info.run_id}")

mlflow.set_tag("mlflow.user", MYNAME)
mlflow.set_tag("simulation", EXEC_NAME)
mlflow.set_tag("run_script", RUN_SCRIPT)
#mlflow.log_params({'git_commit': GIT_INFO['commit'], 'git_branch': GIT_INFO['branch']})
#mlflow.log_artifact('git.diff.txt', artifact_path='git_info')

if USE_MPI:
    mlflow.set_tags({
        "USE_MPI": USE_MPI,
        "MPI_NP": MPI_NP,
    })

In [ ]:
##
## source 状態の記録
##
import utils.hg_utils
from utils.git_utils import get_git_info

local_files = [{"src": "./src"}]
for local_file in local_files:
    repo_path = list(local_file.values())[0]
    repo_name = list(local_file.keys())[0]
    hg_infos = utils.hg_utils.get_hg_info(repo_path, include_diff=True)
    mlflow.log_params({
        f"{repo_name}_branch": hg_infos["branch"],
        f"{repo_name}_revision_full": hg_infos["revision_full"],
        f"{repo_name}_rev": hg_infos["rev"],
    })

    hg_diff = hg_infos["diff"]
    if hg_diff:
        diff_file = f"{repo_name}.diff.txt"
        with open(diff_file, "w") as f:
            f.write(hg_diff)
        mlflow.log_artifact(diff_file, artifact_path="diff")
        os.remove(diff_file)


git_branch, git_commit_hash, git_diff = get_git_info()
mlflow.log_params({
    "lume_branch": git_branch,
    "lume_commit_hash": git_commit_hash,
})
if git_diff:
    git_diff_file = f"git.diff.txt"
    with open(git_diff_file, "w") as f:
        f.write(git_diff)
    mlflow.log_artifact(git_diff_file, artifact_path="diff")
    os.remove(git_diff_file)


In [ ]:
np = MPI_NP if USE_MPI else 1

batch_script = f'''#! /bin/sh
#SBATCH -J {RUN_NAME}
#SBATCH -o {RUN_NAME}.out.txt
#SBATCH -e {RUN_NAME}.err
#SBATCH --nodes=1
#SBATCH --ntasks-per-node={np}
#SBATCH -t 00:00:00

export RUN_SCRIPT={RUN_SCRIPT}
export MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI}
export RUN_NAME={RUN_NAME}
export ARCHIVE_COMMAND={ARCHIVE_COMMAND}
export PREV_RUNID={PREV_RUNID}
export RESTART_FROM={RESTART_FROM}
# JSON 文字列はシェルで空白区切りされないようにクォートする
export SIM_PARAMS='{json.dumps(sim_params)}'
export DUMP_FILES='{json.dumps(dump_files)}'
export SNAPSHOTS={snapshots}
export RESTART_FILES='{json.dumps(restart_files)}'
export ARTIFACTS_CLEANUP={artifacts_cleanup}

if [ {USE_MPI} = "True" ];
then
    mpirun -np {np} python3 Run.py {mlflow_run.info.run_id}
else
    python3 Run.py {mlflow_run.info.run_id}
fi

if [ $? -ne 0 ];
then
    echo "Run.py aborted"
    python3 Cleanup.py {mlflow_run.info.run_id}
fi

'''

import subprocess

# sbatch コマンドに batch_script を標準入力で渡して実行
proc = subprocess.run(
    ["sbatch"],
    input=batch_script,
    text=True,
    capture_output=True
)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)

JOBN = proc.stdout.replace('Submitted batch job ', '').strip()

In [ ]:
os.system(f'squeue -u {MYNAME}')

In [ ]:
raise Exception("一旦ここで停止")

---
### 5. 以下は中断処理

In [ ]:
os.system(f'scancel {JOBN}')

In [ ]:
mlflow.log_artifact(f'{RUN_NAME}.out', artifact_path='output')
mlflow.log_artifact(f'{RUN_NAME}.err', artifact_path='output')

In [ ]:
artifacts = {
    "dumpfiles": dump_files,
    "snapshots": [snapshots],
    "restarts": restart_files,
}
if "log_file" in sim_params:
    artifacts.update({"log": f'log.{sim_params["log_file"]}'})

sim.store_artifacts(artifacts, mlflow.log_artifact, f"{ARCHIVE_COMMAND} -t", cleanup=artifacts_cleanup)

In [ ]:
## 異常・中断時の mlflow.end_run()
run_info = mlflow.get_run(mlflow_run.info.run_id)
if run_info.info.lifecycle_stage == "active":
    mlflow.end_run(status='KILLED')


---
### 6. cleanup

In [ ]:
## slurm の出力ファイルを手動で削除
os.system(f'rm -rf {RUN_NAME}.out.txt')
os.system(f'rm -rf {RUN_NAME}.err')